# Predicting Median Graduate Student Debt — College Scorecard

**Mission & problem.** Make higher education financially transparent so students are not trapped by debt.
**Use case.** Given an institution's profile (cost, size, aid mix, outcomes), predict the **median federal loan debt of its graduates** (`GRAD_DEBT_MDN`, USD).
**Who uses it.** Prospective students and families comparing schools, and college advisors / policymakers flagging institutions whose debt burden is out of line with their profile.

**Dataset.** U.S. Department of Education *College Scorecard — Most Recent Cohorts, Institution level* (6,272 institutions × 3,379 columns). Source: https://collegescorecard.ed.gov/data/

## 1. Load candidate columns
The raw file has 3,379 columns; we load only a candidate set chosen from the Scorecard data dictionary. Scorecard marks small-cohort values as `PrivacySuppressed`, which we treat as missing.

In [1]:
import numpy as np
import pandas as pd

RAW = "Most-Recent-Cohorts-Institution.csv"
TARGET = "GRAD_DEBT_MDN"
CANDIDATES = [TARGET, "CONTROL", "PREDDEG", "REGION", "LOCALE", "STABBR",
              "ADM_RATE", "SAT_AVG", "C150_4", "RET_FT4", "COSTT4_A",
              "UGDS", "TUITIONFEE_IN", "TUITIONFEE_OUT", "PCTPELL", "PCTFLOAN",
              "UG25ABV", "INEXPFTE", "MD_EARN_WNE_P10", "AVGFACSAL"]

raw = pd.read_csv(RAW, usecols=CANDIDATES, low_memory=False).replace("PrivacySuppressed", np.nan)
num_cols = [c for c in CANDIDATES if c != "STABBR"]
raw[num_cols] = raw[num_cols].apply(pd.to_numeric, errors="coerce")
raw = raw[raw[TARGET].notna()]
print(raw.shape)
raw.isna().mean().sort_values(ascending=False).round(2)

(4776, 20)


SAT_AVG            0.79
ADM_RATE           0.65
RET_FT4            0.61
C150_4             0.58
COSTT4_A           0.42
AVGFACSAL          0.35
TUITIONFEE_OUT     0.35
TUITIONFEE_IN      0.35
UG25ABV            0.10
UGDS               0.10
INEXPFTE           0.09
PCTPELL            0.09
PCTFLOAN           0.09
LOCALE             0.09
MD_EARN_WNE_P10    0.05
STABBR             0.00
REGION             0.00
CONTROL            0.00
PREDDEG            0.00
GRAD_DEBT_MDN      0.00
dtype: float64

## 2. Feature selection — what we drop and why

| Dropped | Reason |
|---|---|
| IDs, names, URLs, ~3,350 other columns | identifiers / irrelevant / not in candidate set |
| Every other debt column (`DEBT_MDN`, `GRAD_DEBT_MDN10YR`, repayment rates…) | **target leakage** — they are the target in another form |
| `ADM_RATE`, `SAT_AVG`, `C150_4`, `RET_FT4`, `COSTT4_A` | 42–79 % missing; would shrink the data below a useful size |
| `TUITIONFEE_OUT` | near-duplicate of `TUITIONFEE_IN` (multicollinear) |
| `LOCALE`, `STABBR` | high-cardinality geography already summarised by `REGION` |

**Kept (11 features):** numeric — `UGDS` (enrolment), `TUITIONFEE_IN`, `PCTPELL`, `PCTFLOAN`, `UG25ABV` (share of students over 25), `INEXPFTE` (instructional spend per student), `MD_EARN_WNE_P10` (median earnings 10 yrs after entry), `AVGFACSAL`; categorical — `CONTROL` (1 public, 2 private non-profit, 3 for-profit), `PREDDEG` (predominant degree 1–3), `REGION` (0–9).

In [2]:
NUMERIC = ["UGDS", "TUITIONFEE_IN", "PCTPELL", "PCTFLOAN", "UG25ABV",
           "INEXPFTE", "MD_EARN_WNE_P10", "AVGFACSAL"]
CATEGORICAL = ["CONTROL", "PREDDEG", "REGION"]
FEATURES = NUMERIC + CATEGORICAL

df = raw[FEATURES + [TARGET]].dropna()
df = df[df["PREDDEG"].between(1, 3) & df["REGION"].between(1, 9)]  # drop 'not classified' / service-academy codes
df[CATEGORICAL + ["UGDS"]] = df[CATEGORICAL + ["UGDS"]].astype(int)
print(df.shape)
assert len(df) >= 1000 and len(FEATURES) >= 8 and df.isna().sum().sum() == 0
df.describe().T

(2699, 12)


,count,mean,std,min,25%,50%,75%,max
UGDS,2699.0,4827.963320,8648.216031,1.0000,841.00000,1958.0000,5116.5000,163164.0
TUITIONFEE_IN,2699.0,20522.631345,18419.597160,1104.0000,6260.00000,13641.0000,31933.5000,75162.0
PCTPELL,2699.0,0.401974,0.175084,0.0018,0.27465,0.3758,0.4890,1.0
PCTFLOAN,2699.0,0.391440,0.250103,0.0000,0.16470,0.3828,0.5864,1.0
UG25ABV,2699.0,0.247027,0.223914,0.0005,0.07340,0.1847,0.3314,1.0
INEXPFTE,2699.0,10699.997406,9515.685189,200.0000,6184.00000,8671.0000,12398.5000,181929.0
MD_EARN_WNE_P10,2699.0,55399.699889,16514.101066,20666.0000,44124.00000,51652.0000,63476.0000,166642.0
AVGFACSAL,2699.0,8470.645795,2809.423430,729.0000,6628.50000,8043.0000,9787.0000,25354.0
CONTROL,2699.0,1.603927,0.665689,1.0000,1.00000,2.0000,2.0000,3.0
PREDDEG,2699.0,2.457947,0.737630,1.0000,2.00000,3.0000,3.0000,3.0


## 3. Save the cleaned subset
The raw CSV (102 MB) exceeds GitHub's limit, so we commit this small cleaned file. The API's `/retrain` endpoint reuses it as the original training data.

In [3]:
df.to_csv("../API/debt_data.csv", index=False)
pd.read_csv("../API/debt_data.csv").head()

,UGDS,TUITIONFEE_IN,PCTPELL,PCTFLOAN,UG25ABV,INEXPFTE,MD_EARN_WNE_P10,AVGFACSAL,CONTROL,PREDDEG,REGION,GRAD_DEBT_MDN
0,6124,10558.0,0.6419,0.7714,0.0707,7254.0,41266.0,8699.0,1,3,5,27204.0
1,11635,9812.0,0.3885,0.4100,0.1656,17855.0,59631.0,12612.0,1,3,5,21250.0
2,6591,12458.0,0.2781,0.3316,0.1511,9877.0,72098.0,10639.0,1,3,5,20550.0
3,3477,11248.0,0.7197,0.7895,0.0692,10723.0,40585.0,8153.0,1,3,5,28197.0
4,33227,12484.0,0.2147,0.3528,0.0601,9728.0,66081.0,11419.0,1,3,5,21500.0
